In [1]:
"""
Optimized 6DoF Pick-and-Place in PyBullet - Refined Version (GraspNet Integrated)
- Handles object tracking: skips objects that fall off the table.
- Refined IK and joint interpolation for more precise grasping.
- Loads GraspNet checkpoint for potential 6DoF candidate generation.
"""

import time
import random
import numpy as np
import pybullet as p
import pybullet_data
from pathlib import Path

# =====================================================
# 1. CONFIG
# =====================================================
GUI = True
REALTIME_SLEEP = True
SIM_HZ = 240
DT = 1.0 / SIM_HZ

NUM_OBJECTS = 12
MAX_GRASP_ATTEMPTS = 3
GRASP_ASSIST = True

# Workspace boundaries
TABLE_TOP_Z = 0.14
TABLE_THICKNESS = 0.08
TABLE_CENTER = np.array([0.57, -0.13, TABLE_TOP_Z - TABLE_THICKNESS / 2])
TABLE_HALF_EXTENTS = np.array([0.34, 0.28, TABLE_THICKNESS / 2])

# Safety and Height Params
SAFE_Z_ABOVE_TABLE = 0.015
MIN_GRASP_LIFT_Z = TABLE_TOP_Z + 0.08
PRE_GRASP_HEIGHT = 0.15
LIFT_HEIGHT = 0.25

# Panda Robot Config
ARM_JOINTS = list(range(7))
FINGER_JOINTS = [9, 10]
FINGER_LINKS = [9, 10]
EE_LINK_INDEX = 11

HOME_JOINTS = [0.0, -0.45, 0.0, -2.35, 0.0, 1.95, 0.78]
REST_JOINTS = HOME_JOINTS[:]

PANDA_LOWER_LIMITS = [-2.8973, -1.7628, -2.8973, -3.0718, -2.8973, -0.0175, -2.8973]
PANDA_UPPER_LIMITS = [ 2.8973,  1.7628,  2.8973, -0.0698,  2.8973,  3.7525,  2.8973]
PANDA_JOINT_RANGES = [u - l for l, u in zip(PANDA_LOWER_LIMITS, PANDA_UPPER_LIMITS)]

# =====================================================
# 2. GRASPNET INTEGRATION
# =====================================================
CHECKPOINT_PATH = r"d:\CV_2026\checkpoint-rs.zip"
LOAD_CHECKPOINT = True

def load_graspnet():
    if not LOAD_CHECKPOINT:
        return None

    path = Path(CHECKPOINT_PATH)
    if not path.exists():
        print("Không tìm thấy checkpoint:", path)
        return None

    try:
        import torch
        # We try to import the model class. This assumes the environment is set up.
        from models.graspnet import GraspNet
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        net = GraspNet(input_feature_dim=0, num_view=300, num_angle=12, num_depth=4,
                       cylinder_radius=0.05, hmin=-0.02, hmax_list=[0.01, 0.02, 0.03, 0.04], is_training=False)
        
        checkpoint = torch.load(str(path), map_location=device)
        state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
        net.load_state_dict(state_dict, strict=False)
        net.to(device)
        net.eval()
        print("Load checkpoint GraspNet OK:", path)
        return net
    except Exception as e:
        print("Load checkpoint lỗi:", type(e).__name__, e)
        return None

# Load the model at startup
model_net = load_graspnet()

# =====================================================
# 3. PYBULLET INIT
# =====================================================
if p.isConnected():
    p.disconnect()

p.connect(p.GUI if GUI else p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.setTimeStep(DT)

p.resetDebugVisualizerCamera(
    cameraDistance=1.8,
    cameraYaw=45,
    cameraPitch=-30,
    cameraTargetPosition=[0.55, -0.1, 0.2],
)

def step_sim(steps=1):
    for _ in range(steps):
        p.stepSimulation()
        if REALTIME_SLEEP:
            time.sleep(DT)

# =====================================================
# 4. CREATE SCENE
# =====================================================
plane_id = p.loadURDF("plane.urdf")

def create_static_box(position, half_extents, color):
    col = p.createCollisionShape(p.GEOM_BOX, halfExtents=half_extents)
    vis = p.createVisualShape(p.GEOM_BOX, halfExtents=half_extents, rgbaColor=color)
    return p.createMultiBody(0, col, vis, position)

object_platform = create_static_box(
    TABLE_CENTER, TABLE_HALF_EXTENTS, [0.5, 0.5, 0.5, 1.0]
)

robot_id = p.loadURDF("franka_panda/panda.urdf", [0, 0, 0], useFixedBase=True)

def reset_robot():
    for j, q in zip(ARM_JOINTS, HOME_JOINTS):
        p.resetJointState(robot_id, j, q)
    p.resetJointState(robot_id, 9, 0.04)
    p.resetJointState(robot_id, 10, 0.04)
    step_sim(50)

reset_robot()

# Bin for placing objects (with walls)
bin_center = np.array([0.5, 0.4, 0.05])
bin_size = [0.3, 0.3, 0.1] # Width, Depth, Height
bin_wall_thick = 0.01

# Bottom
create_static_box(bin_center, [bin_size[0]/2, bin_size[1]/2, bin_wall_thick], [0.2, 0.2, 0.8, 0.5])
# Walls
create_static_box(bin_center + [bin_size[0]/2, 0, bin_size[2]/2], [bin_wall_thick, bin_size[1]/2, bin_size[2]/2], [0.2, 0.2, 0.8, 0.5])
create_static_box(bin_center - [bin_size[0]/2, 0, bin_size[2]/2], [bin_wall_thick, bin_size[1]/2, bin_size[2]/2], [0.2, 0.2, 0.8, 0.5])
create_static_box(bin_center + [0, bin_size[1]/2, bin_size[2]/2], [bin_size[0]/2, bin_wall_thick, bin_size[2]/2], [0.2, 0.2, 0.8, 0.5])
create_static_box(bin_center - [0, bin_size[1]/2, bin_size[2]/2], [bin_size[0]/2, bin_wall_thick, bin_size[2]/2], [0.2, 0.2, 0.8, 0.5])

# =====================================================
# 5. OBJECT MANAGEMENT
# =====================================================
object_ids = []

def create_random_object():
    x = random.uniform(TABLE_CENTER[0] - 0.2, TABLE_CENTER[0] + 0.2)
    y = random.uniform(TABLE_CENTER[1] - 0.15, TABLE_CENTER[1] + 0.15)
    z = TABLE_TOP_Z + 0.05
    
    obj_type = random.choice(["box", "cylinder", "sphere", "book", "bottle"])
    is_transparent = random.random() < 0.3
    color = [random.random(), random.random(), random.random(), 0.3 if is_transparent else 1.0]

    if obj_type in ["box", "book"]:
        he = [random.uniform(0.02, 0.05), random.uniform(0.02, 0.04), random.uniform(0.01, 0.03)]
        col = p.createCollisionShape(p.GEOM_BOX, halfExtents=he)
        vis = p.createVisualShape(p.GEOM_BOX, halfExtents=he, rgbaColor=color)
        mass = 0.1
    elif obj_type == "cylinder": # Like a can
        radius, height = random.uniform(0.02, 0.03), random.uniform(0.05, 0.08)
        col = p.createCollisionShape(p.GEOM_CYLINDER, radius=radius, height=height)
        vis = p.createVisualShape(p.GEOM_CYLINDER, radius=radius, length=height, rgbaColor=color)
        mass = 0.15
    elif obj_type == "bottle":
        radius, height = random.uniform(0.015, 0.025), random.uniform(0.08, 0.12)
        col = p.createCollisionShape(p.GEOM_CYLINDER, radius=radius, height=height)
        vis = p.createVisualShape(p.GEOM_CYLINDER, radius=radius, length=height, rgbaColor=color)
        mass = 0.12
    else: # Sphere / Ball
        radius = random.uniform(0.02, 0.03)
        col = p.createCollisionShape(p.GEOM_SPHERE, radius=radius)
        vis = p.createVisualShape(p.GEOM_SPHERE, radius=radius, rgbaColor=color)
        mass = 0.05

    body = p.createMultiBody(mass, col, vis, [x, y, z])
    p.changeDynamics(body, -1, lateralFriction=1.2, rollingFriction=0.02, spinningFriction=0.02)
    return body

for i in range(NUM_OBJECTS):
    object_ids.append(create_random_object())
step_sim(100)

def is_object_on_table(obj_id):
    pos, _ = p.getBasePositionAndOrientation(obj_id)
    dx = abs(pos[0] - TABLE_CENTER[0])
    dy = abs(pos[1] - TABLE_CENTER[1])
    return dx < (TABLE_HALF_EXTENTS[0] + 0.05) and \
           dy < (TABLE_HALF_EXTENTS[1] + 0.05) and \
           pos[2] > (TABLE_TOP_Z - 0.05)

# =====================================================
# 6. ROBOT CONTROL
# =====================================================
def open_gripper():
    for j in FINGER_JOINTS:
        p.setJointMotorControl2(robot_id, j, p.POSITION_CONTROL, 0.04, force=100)

def close_gripper():
    for j in FINGER_JOINTS:
        p.setJointMotorControl2(robot_id, j, p.POSITION_CONTROL, 0.0, force=200)

def compute_ik(pos, orn):
    return p.calculateInverseKinematics(
        robot_id, EE_LINK_INDEX, pos, orn,
        lowerLimits=PANDA_LOWER_LIMITS,
        upperLimits=PANDA_UPPER_LIMITS,
        jointRanges=PANDA_JOINT_RANGES,
        restPoses=REST_JOINTS,
        maxNumIterations=200,
        residualThreshold=1e-5
    )[:7]

def move_to_joints(target_joints, steps=100):
    start_joints = [p.getJointState(robot_id, j)[0] for j in ARM_JOINTS]
    for i in range(steps):
        t_linear = (i + 1) / steps
        # Cosine S-curve interpolation for smooth acceleration and deceleration
        t_smooth = (1.0 - np.cos(t_linear * np.pi)) / 2.0
        curr_q = [s + (e - s) * t_smooth for s, e in zip(start_joints, target_joints)]
        p.setJointMotorControlArray(
            robot_id, ARM_JOINTS, p.POSITION_CONTROL,
            targetPositions=curr_q, forces=[200]*7
        )
        step_sim(1)

def move_ee(pos, orn, steps=60):
    target_q = compute_ik(pos, orn)
    move_to_joints(target_q, steps)

# =====================================================
# 7. GRASPING LOGIC
# =====================================================
def get_grasp_candidates(obj_id):
    pos, orn = p.getBasePositionAndOrientation(obj_id)
    _, _, yaw = p.getEulerFromQuaternion(orn)
    candidates = []
    # Try different yaws (0, 90 deg relative to object)
    for dyaw in [0, np.pi/2]:
        target_yaw = yaw + dyaw
        target_orn = p.getQuaternionFromEuler([np.pi, 0, target_yaw])
        candidates.append((np.array(pos), target_orn))
    return candidates

def pick_and_place(obj_id):
    if not is_object_on_table(obj_id):
        print(f"Object {obj_id} fell off the table. Skipping.")
        return False

    candidates = get_grasp_candidates(obj_id)
    for grasp_pos, grasp_orn in candidates:
        pre_grasp = grasp_pos + np.array([0, 0, PRE_GRASP_HEIGHT])
        open_gripper()
        move_ee(pre_grasp, grasp_orn, steps=150)
        
        if not is_object_on_table(obj_id):
            print("Object moved during approach!")
            return False
            
        curr_pos, _ = p.getBasePositionAndOrientation(obj_id)
        grasp_pos[2] = curr_pos[2] + 0.005 
        move_ee(grasp_pos, grasp_orn, steps=80)
        step_sim(20)
        
        close_gripper()
        step_sim(60)
        
        lift_pos = grasp_pos + np.array([0, 0, LIFT_HEIGHT])
        move_ee(lift_pos, grasp_orn, steps=150)
        
        curr_pos, _ = p.getBasePositionAndOrientation(obj_id)
        if curr_pos[2] > MIN_GRASP_LIFT_Z:
            print("Lift successful!")
            cid = None
            if GRASP_ASSIST:
                cid = p.createConstraint(robot_id, EE_LINK_INDEX, obj_id, -1, p.JOINT_FIXED, [0,0,0], [0,0,0], [0,0,0])
            
            place_pos = bin_center + np.array([0, 0, 0.2])
            move_ee(place_pos, grasp_orn, steps=200)
            
            if cid: p.removeConstraint(cid)
            open_gripper()
            step_sim(100)
            # Move back smoothly instead of instant teleportation
            move_to_joints(HOME_JOINTS, steps=150)
            return True
        else:
            print("Grasp failed, retrying...")
            open_gripper()
            move_ee(pre_grasp, grasp_orn, steps=80)
    return False

# =========================
# 8. MAIN EXECUTION
# =========================
def run_all():
    random.shuffle(object_ids)
    for obj_id in object_ids:
        print(f"Attempting to pick object {obj_id}...")
        pick_and_place(obj_id)
        
if __name__ == "__main__":
    print(f"Starting Optimized Pick and Place... Model Loaded: {model_net is not None}")
    run_all()
    print("Done.")
    try:
        while True:
            step_sim(1)
    except Exception:
        print('Da dong cua so PyBullet GUI. Dung mo phong.')


Load checkpoint GraspNet OK: d:\CV_2026\checkpoint-rs.zip
Starting Optimized Pick and Place... Model Loaded: True
Attempting to pick object 19...
Lift successful!
Attempting to pick object 17...
Lift successful!
Attempting to pick object 11...
Lift successful!
Attempting to pick object 13...
Lift successful!
Attempting to pick object 9...
Lift successful!
Attempting to pick object 14...
Object 14 fell off the table. Skipping.
Attempting to pick object 8...
Lift successful!
Attempting to pick object 18...
Lift successful!
Attempting to pick object 16...
Grasp failed, retrying...
Grasp failed, retrying...
Attempting to pick object 12...
Object 12 fell off the table. Skipping.
Attempting to pick object 15...
Object 15 fell off the table. Skipping.
Attempting to pick object 10...
Lift successful!
Done.
Da dong cua so PyBullet GUI. Dung mo phong.
